# Tétel 02 — Normális adatmátrix, Kronecker-szorzat, AX̃B

**Cél:** Kézzel, csak numpy-jal megérteni mi történik mátrix szinten.  
Generálunk random normális adatmátrixot, kiszámítjuk a tapasztalati statisztikákat, megvizsgáljuk a Kronecker-szorzat struktúrát, és ellenőrizzük az $A\tilde{X}B$ feltételeket számokkal.

---
Jelölések:
- $\tilde{X}$ — $n\times p$-s normális adatmátrix (1 sor = 1 megfigyelés)
- $n$ = megfigyelések száma, $p$ = változók száma
- $H = I_n - \frac{1}{n}\mathbf{1}_n\mathbf{1}_n^T$ — centráló mátrix

In [1]:
import numpy as np
import pandas as pd

np.random.seed(7)
print("Kész.")

Kész.


## 1. Normális adatmátrix generálása

$$\tilde{X} = \begin{pmatrix} \underline{\xi}_1^T \\ \vdots \\ \underline{\xi}_n^T \end{pmatrix}, \quad \underline{\xi}_i \overset{\text{FAE}}{\sim} N_p(\underline{\mu}, \Sigma)$$

Trükk: ha $Z \sim N(0, I)$, akkor $ZL^T + \mathbf{1}_n\underline{\mu}^T \sim N_p(\underline{\mu},\Sigma)$ adatmátrix, ahol $\Sigma = LL^T$.

In [2]:
n = 200  # megfigyelések száma
p = 3    # változók száma

mu = np.array([1.0, 3.0, -2.0])

Sigma = np.array([
    [4.0, 1.2, 0.5],
    [1.2, 2.0, 0.3],
    [0.5, 0.3, 1.5]
])

L = np.linalg.cholesky(Sigma)  # Sigma = L @ L.T

Z_std = np.random.randn(n, p)          # n×p standard normális
X = Z_std @ L.T + mu                   # n×p normális adatmátrix

print(f"Adatmátrix X mérete: {X.shape}  ({n} sor = megfigyelés, {p} oszlop = változó)")
print("\nElső 5 sor:")
print(X[:5].round(3))

Adatmátrix X mérete: (200, 3)  (200 sor = megfigyelés, 3 oszlop = változó)

Első 5 sor:
[[ 4.381  3.418 -1.593]
 [ 1.815  2.234 -1.988]
 [ 0.998  0.752 -0.991]
 [ 2.201  2.559 -2.128]
 [ 2.011  2.968 -2.194]]


## 2. Centráló mátrix $H$ és tapasztalati statisztikák

$$H = I_n - \frac{1}{n}\mathbf{1}_n\mathbf{1}_n^T$$

Tulajdonságok: $H^2 = H$ (idempotens), $H^T = H$ (szimmetrikus), $H\mathbf{1}_n = \mathbf{0}$.

$$\bar{\underline{\xi}} = \frac{1}{n}\tilde{X}^T\mathbf{1}_n, \qquad S = \frac{1}{n}\tilde{X}^T H \tilde{X}$$

In [3]:
ones = np.ones(n)                           # 1_n vektor
H = np.eye(n) - np.outer(ones, ones) / n   # n×n centráló mátrix

# Ellenőrzések
print("H idempotens? H@H == H:", np.allclose(H @ H, H))
print("H szimmetrikus?", np.allclose(H, H.T))
print("H @ 1_n == 0?", np.allclose(H @ ones, 0))

H idempotens? H@H == H: True
H szimmetrikus? True
H @ 1_n == 0? True


In [4]:
# Tapasztalati átlag: (1/n) * X.T @ 1_n  -- azaz oszlopátlag
xi_bar = (X.T @ ones) / n

# Tapasztalati szórásmátrix: (1/n) * X.T @ H @ X
S = (X.T @ H @ X) / n

print("Tapasztalati átlag (becsült mu):")
print(xi_bar.round(4))
print("Valódi mu:")
print(mu)

print("\nTapasztalati szórásmátrix S = (1/n) X.T @ H @ X:")
print(S.round(4))
print("\nValódi Sigma:")
print(Sigma)

Tapasztalati átlag (becsült mu):
[ 0.9369  2.9373 -2.1217]
Valódi mu:
[ 1.  3. -2.]

Tapasztalati szórásmátrix S = (1/n) X.T @ H @ X:
[[3.7357 0.9268 0.3618]
 [0.9268 1.7023 0.1342]
 [0.3618 0.1342 1.4974]]

Valódi Sigma:
[[4.  1.2 0.5]
 [1.2 2.  0.3]
 [0.5 0.3 1.5]]


## 3. Kronecker-szorzat kézzel

$$A \otimes B = \begin{pmatrix} a_{11}B & a_{12}B & \cdots \\ a_{21}B & \ddots & \\ \end{pmatrix}$$

Ellenőrizzük az adatmátrix karakterizációját: $\tilde{X}^V \sim N_{np}(\underline{\mu}\otimes\mathbf{1}_n, \Sigma\otimes I_n)$.

In [5]:
# Kronecker-szorzat kézzel (kis mátrixon)
def kronecker(A, B):
    """A ⊗ B kézzel."""
    m, n = A.shape
    p, q = B.shape
    K = np.zeros((m*p, n*q))
    for i in range(m):
        for j in range(n):
            K[i*p:(i+1)*p, j*q:(j+1)*q] = A[i, j] * B
    return K

# Kis példa: 2×2 ⊗ 2×2
A_k = np.array([[1, 2], [3, 4]])
B_k = np.array([[0, 5], [6, 7]])

K_kezzel = kronecker(A_k, B_k)
K_numpy  = np.kron(A_k, B_k)

print("A ⊗ B (kézzel):")
print(K_kezzel)
print("\nEllenőrzés: np.kron egyezik?")
print(np.allclose(K_kezzel, K_numpy))

A ⊗ B (kézzel):
[[ 0.  5.  0. 10.]
 [ 6.  7. 12. 14.]
 [ 0. 15.  0. 20.]
 [18. 21. 24. 28.]]

Ellenőrzés: np.kron egyezik?
True


In [6]:
# Vektorizálás: X^V = oszlopok egymás alá
# X.flatten('F') = Fortran-order = oszlopok egymás alá (ez a matematikai konvenció)
X_vec = X.flatten('F')   # np*1 vektor

print(f"X^V mérete: {X_vec.shape}  (= n*p = {n*p})")

# A vektorizált adatmátrix várható értéke: mu ⊗ 1_n
mu_vec_elmeleti = np.kron(mu, ones)   # p*n méretű vektor

print(f"\nmu ⊗ 1_n mérete: {mu_vec_elmeleti.shape}")
print("Elméleti E[X^V] első 10 eleme:", mu_vec_elmeleti[:10].round(3))
print("Tényleges X^V átlag első 10 eleme:", X_vec[:10].mean(), "(ez egy elem, nem átlag)")
print("X^V empirikus átlag:", X_vec.mean().round(4), "(összes elem átlaga -- nem informatív)")

# Helyesebb: az elméleti átlag minden pozícióban mu_k értéke
print("\nElső 6 elméleti átlag (mu_1, mu_1, ..., mu_2, mu_2, ...):",
      mu_vec_elmeleti[:6].round(3))

X^V mérete: (600,)  (= n*p = 600)

mu ⊗ 1_n mérete: (600,)
Elméleti E[X^V] első 10 eleme: [1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
Tényleges X^V átlag első 10 eleme: 0.968960481189679 (ez egy elem, nem átlag)
X^V empirikus átlag: 0.5842 (összes elem átlaga -- nem informatív)

Első 6 elméleti átlag (mu_1, mu_1, ..., mu_2, mu_2, ...): [1. 1. 1. 1. 1. 1.]


In [7]:
# A vektorizált kovmátrix struktúrája: Sigma ⊗ I_n
# Mérete: np×np = 600×600, csak egy blokkot nézünk meg

Sigma_kron_In = np.kron(Sigma, np.eye(n))
print(f"Sigma ⊗ I_n mérete: {Sigma_kron_In.shape}")

# Felső bal blokk: Sigma[0,0] * I_n -- az első változó önvariancia-blokkja
blokk_00 = Sigma_kron_In[:n, :n]
print(f"\nFelső-bal {n}×{n} blokk = Sigma[0,0]*I_n = {Sigma[0,0]}*I:")
print(f"  Átló első 5 elem: {np.diag(blokk_00)[:5].round(4)}")
print(f"  Off-átló elem [0,1]: {blokk_00[0,1]}  (kell = 0, mert a sorok függetlenek)")

# Kereszt-blokk Sigma[0,1]*I_n -- az 1. és 2. változó kereszt-kovariancia blokkja
blokk_01 = Sigma_kron_In[:n, n:2*n]
print(f"\nKeresztes blokk 01 átlója = Sigma[0,1]*I = {Sigma[0,1]}:")
print(f"  Átló első 5 elem: {np.diag(blokk_01)[:5].round(4)}")

Sigma ⊗ I_n mérete: (600, 600)

Felső-bal 200×200 blokk = Sigma[0,0]*I_n = 4.0*I:
  Átló első 5 elem: [4. 4. 4. 4. 4.]
  Off-átló elem [0,1]: 0.0  (kell = 0, mert a sorok függetlenek)

Keresztes blokk 01 átlója = Sigma[0,1]*I = 1.2:
  Átló első 5 elem: [1.2 1.2 1.2 1.2 1.2]


## 4. ★ Mikor lesz $A\tilde{X}B$ is normális adatmátrix?

**Feltételek:** $\tilde{Y} = A\tilde{X}B$ akkor és csak akkor normális adatmátrix, ha:
- **(i)** $A\mathbf{1}_n = \alpha\mathbf{1}_m$ &nbsp;VAGY&nbsp; $B^T\underline{\mu} = \mathbf{0}$
- **(ii)** $AA^T = \beta I_m$ &nbsp;VAGY&nbsp; $B^T\Sigma B = 0$

Ellenőrizzük számokkal konkrét $A$, $B$ választásokra.

In [8]:
# --- Példa 1: A = (1/n) * 1_n^T (átlagolás), B = I_p ---
# Ekkor Y = (1/n) * 1^T @ X = xi_bar^T (1×p, a tapasztalati átlag)

A1 = ones.reshape(1, n) / n   # 1×n
B1 = np.eye(p)                 # p×p

# (i) ellenőrzés: A @ 1_n = alpha * 1_m ?
A1_times_1 = A1 @ ones
print("=== Példa 1: A = (1/n)*1^T, B = I ===")
print(f"(i) A @ 1_n = {A1_times_1}  ->  alpha = {A1_times_1[0]:.4f}, 1_m = [1] --> TELJESÜL")

# (ii) ellenőrzés: A @ A.T = beta * I_m ?
A1A1T = A1 @ A1.T
beta1 = A1A1T[0, 0]
print(f"(ii) A @ A.T = {A1A1T}  ->  beta = {beta1:.6f} = 1/n --> TELJESÜL")

Y1 = A1 @ X @ B1
print(f"\nY = A @ X @ B mérete: {Y1.shape}  (ez a tapasztalati átlagvektor^T)")
print(f"Y értéke: {Y1.flatten().round(4)}")
print(f"Elméleti E[Y]: mu^T = {mu}")

=== Példa 1: A = (1/n)*1^T, B = I ===
(i) A @ 1_n = [1.]  ->  alpha = 1.0000, 1_m = [1] --> TELJESÜL
(ii) A @ A.T = [[0.005]]  ->  beta = 0.005000 = 1/n --> TELJESÜL

Y = A @ X @ B mérete: (1, 3)  (ez a tapasztalati átlagvektor^T)
Y értéke: [ 0.9369  2.9373 -2.1217]
Elméleti E[Y]: mu^T = [ 1.  3. -2.]


In [9]:
# --- Példa 2: A = H (centráló), B = I_p ---
# Ekkor Y = H @ X -- a centrált adatmátrix

A2 = H        # n×n
B2 = np.eye(p)

# (i) A @ 1_n = alpha * 1_m ? Vagy B^T @ mu = 0 ?
A2_times_1 = A2 @ ones
print("=== Példa 2: A = H (centráló), B = I ===")
print(f"(i) A @ 1_n = H @ 1_n = 0-vektor (max abs = {np.abs(A2_times_1).max():.2e})")
print("    Ez alpha * 1_n lenne, de alpha = 0, 1_n mérete OK --> TELJESÜL (alpha=0)")

# (ii) A @ A.T = beta * I_m ?
A2A2T = A2 @ A2.T
# H szimmetrikus idempotens: H @ H.T = H @ H = H
is_beta_I = np.allclose(A2A2T, A2A2T[0,0] * np.eye(n))
print(f"\n(ii) A @ A.T = H @ H.T = H @ H = H")
print(f"     Ez beta*I_n lenne -- de H NEM skaláris * I (rangja n-1, nem n)")

# Alternatíva: B^T @ Sigma @ B = 0?
BTSB = B2.T @ Sigma @ B2
print(f"     B^T @ Sigma @ B = I @ Sigma @ I = Sigma, nem 0")
print()
print("KÖVETKEZTETÉS: H @ X @ I NEM garantáltan normális adatmátrix!")
print("(A sorok nem függetlenek -- H összekeveri őket.)")

=== Példa 2: A = H (centráló), B = I ===
(i) A @ 1_n = H @ 1_n = 0-vektor (max abs = 5.55e-16)
    Ez alpha * 1_n lenne, de alpha = 0, 1_n mérete OK --> TELJESÜL (alpha=0)

(ii) A @ A.T = H @ H.T = H @ H = H
     Ez beta*I_n lenne -- de H NEM skaláris * I (rangja n-1, nem n)
     B^T @ Sigma @ B = I @ Sigma @ I = Sigma, nem 0

KÖVETKEZTETÉS: H @ X @ I NEM garantáltan normális adatmátrix!
(A sorok nem függetlenek -- H összekeveri őket.)


In [10]:
# --- Példa 3: A = I_n, B = b vektor (egy kombinációt veszünk) ---
# Vizsgáljuk: X @ b (lineáris kombináció az oszlopokra)
# Ha b^T mu = 0: feltétel (i) második lehetősége

# Választunk b-t úgy, hogy b^T @ mu = 0
# mu = [1, 3, -2], b = [3, -1, 0] -> b^T @ mu = 3 - 3 + 0 = 0
b = np.array([[3.0], [-1.0], [0.0]])   # p×1

print("=== Példa 3: A = I_n, B = b (ahol b^T mu = 0) ===")
print(f"b = {b.flatten()}")
print(f"b^T @ mu = {(b.T @ mu)[0]:.4f}  --> (i) TELJESÜL (második lehetőség)")

A3 = np.eye(n)
# (ii) A @ A.T = I_n @ I_n = I_n = 1 * I_n --> TELJESÜL
print(f"A @ A.T = I_n @ I_n = I_n --> (ii) TELJESÜL (beta=1)")

Y3 = X @ b   # n×1
mu_Y3 = b.T @ mu
Sigma_Y3 = b.T @ Sigma @ b

print(f"\nY = X @ b mérete: {Y3.shape}")
print(f"Elméleti E[Y] = b^T mu = {mu_Y3[0]:.4f}   (= 0)")
print(f"Elméleti Var[Y] = b^T Sigma b = {Sigma_Y3[0,0]:.4f}")
print(f"Tényleges Y átlag: {Y3.mean():.4f}")
print(f"Tényleges Y variancia: {Y3.var():.4f}")

=== Példa 3: A = I_n, B = b (ahol b^T mu = 0) ===
b = [ 3. -1.  0.]
b^T @ mu = 0.0000  --> (i) TELJESÜL (második lehetőség)
A @ A.T = I_n @ I_n = I_n --> (ii) TELJESÜL (beta=1)

Y = X @ b mérete: (200, 1)
Elméleti E[Y] = b^T mu = 0.0000   (= 0)
Elméleti Var[Y] = b^T Sigma b = 30.8000
Tényleges Y átlag: -0.1265
Tényleges Y variancia: 29.7628


## 5. ★ $A\tilde{X}B$ és $C\tilde{X}D$ függetlensége

$$\tilde{Y} \perp \tilde{Z} \iff B^T\Sigma D = 0 \quad \text{vagy} \quad AC^T = 0$$

A klasszikus eset: $\bar{\underline{\xi}}$ és $S$ függetlensége (Fisher–Bartlett).
- $\bar{\underline{\xi}}$ ← $A = \frac{1}{n}\mathbf{1}^T$, $B = I$
- $H\tilde{X}$ ← $C = H$, $D = I$
- Ellenőrzés: $AC^T = \frac{1}{n}\mathbf{1}^T H = ?$

In [11]:
# Fisher-Bartlett: xi_bar és S függetlensége
A_avg = ones.reshape(1, n) / n   # 1×n
B_avg = np.eye(p)                  # p×p

C_H = H                            # n×n
D_H = np.eye(p)                    # p×p

# AC^T = (1/n) * 1^T @ H
ACT = A_avg @ C_H.T

print("AC^T = (1/n) * 1_n^T @ H^T = (1/n) * 1_n^T @ H:")
print(f"Max abs érték: {np.abs(ACT).max():.2e}  --> numerikusan 0")
print("Magyarázat: H @ 1_n = 0, ezért 1^T @ H = (H^T @ 1)^T = (H @ 1)^T = 0^T = 0")
print()
print("TEHÁT: AC^T = 0  -->  xi_bar és H@X FÜGGETLEN  -->  xi_bar és S is független ✓")

AC^T = (1/n) * 1_n^T @ H^T = (1/n) * 1_n^T @ H:
Max abs érték: 7.37e-18  --> numerikusan 0
Magyarázat: H @ 1_n = 0, ezért 1^T @ H = (H^T @ 1)^T = (H @ 1)^T = 0^T = 0

TEHÁT: AC^T = 0  -->  xi_bar és H@X FÜGGETLEN  -->  xi_bar és S is független ✓


In [12]:
# Empirikus szemléltetés: xi_bar és S "függetlensége"
# Sok mintán számítjuk: a xi_bar értéke és az S[0,0] értéke nem korrelál

N_sim = 2000   # szimulációk száma
xi_bar_1_vals = np.zeros(N_sim)   # xi_bar 1. koordinátája
S_11_vals     = np.zeros(N_sim)   # S[0,0] értéke

for k in range(N_sim):
    X_k = np.random.randn(n, p) @ L.T + mu
    xi_bar_1_vals[k] = X_k.mean(axis=0)[0]          # 1. koordináta átlaga
    H_k = np.eye(n) - np.outer(np.ones(n), np.ones(n)) / n
    S_k = (X_k.T @ H_k @ X_k) / n
    S_11_vals[k] = S_k[0, 0]

# Korreláció köztük kell ~0
corr = np.corrcoef(xi_bar_1_vals, S_11_vals)[0, 1]
print(f"Empirikus korreláció xi_bar[0] és S[0,0] között ({N_sim} szimuláción): {corr:.4f}")
print("(Ha közel 0: a függetlenség tényleg fennáll)")

Empirikus korreláció xi_bar[0] és S[0,0] között (2000 szimuláción): -0.0404
(Ha közel 0: a függetlenség tényleg fennáll)


## 6. Blokk-struktúra és függetlenítés

Bontsuk $\tilde{X}$ oszlopait két részre: $\tilde{X} = [\tilde{X}_1 \mid \tilde{X}_2]$, ahol $p_1 = 2$, $p_2 = 1$.

A **függetlenített** részmátrix:
$$\tilde{X}_{2\cdot 1} = \tilde{X}_2 - \tilde{X}_1 \Sigma_{11}^{-1}\Sigma_{12}$$

Ez analóg a normális korreláció tételével — ott ugyanígy "levonjuk" a másik változó lineáris hatását.

In [13]:
p1, p2 = 2, 1

X1 = X[:, :p1]   # n×2  (xi_1, xi_2 változók)
X2 = X[:, p1:]   # n×1  (xi_3 változó)

Sigma11 = Sigma[:p1, :p1]   # 2×2
Sigma12 = Sigma[:p1, p1:]   # 2×1
Sigma22 = Sigma[p1:, p1:]   # 1×1

print("Sigma blokkok:")
print(f"Sigma11 (2×2):\n{Sigma11}")
print(f"Sigma12 (2×1): {Sigma12.flatten()}")
print(f"Sigma22 (1×1): {Sigma22[0,0]}")

Sigma blokkok:
Sigma11 (2×2):
[[4.  1.2]
 [1.2 2. ]]
Sigma12 (2×1): [0.5 0.3]
Sigma22 (1×1): 1.5


In [14]:
# Függetlenített rész: X_{2.1} = X2 - X1 @ Sigma11^{-1} @ Sigma12
Sigma11_inv = np.linalg.inv(Sigma11)
X2_given_1 = X2 - X1 @ Sigma11_inv @ Sigma12   # n×1

# Elméleti: Sigma_{22.1} = Sigma22 - Sigma21 @ Sigma11^{-1} @ Sigma12
Sigma22_given_1 = Sigma22 - Sigma12.T @ Sigma11_inv @ Sigma12

print("Feltételes szórásmátrix Sigma_{22|1} (elméleti):")
print(Sigma22_given_1.round(4))

print("\nX_{2|1} tényleges varianciája (kell egyezni):")
print(X2_given_1.var().round(4))

# Függetlenség ellenőrzése: X1 és X_{2.1} korrelálatlan kell legyen
print("\nKorrelációk X1 oszlopai és X_{2|1} között (kell ~0):")
for i in range(p1):
    corr_i = np.corrcoef(X1[:, i], X2_given_1[:, 0])[0, 1]
    print(f"  corr(X1[:,{i}], X_{{2|1}}): {corr_i:.4f}")

Feltételes szórásmátrix Sigma_{22|1} (elméleti):
[[1.4238]]

X_{2|1} tényleges varianciája (kell egyezni):
1.4686

Korrelációk X1 oszlopai és X_{2|1} között (kell ~0):
  corr(X1[:,0], X_{2|1}): -0.0373
  corr(X1[:,1], X_{2|1}): -0.0708


## 7. Összefoglalás — Vektorizációs azonosság ellenőrzése

$(ACB)^V = (B^T \otimes A) C^V$ — nézzük meg kis mátrixon számokkal.

In [15]:
# (ACB)^V = (B^T ⊗ A) @ C^V
rng = np.random.default_rng(99)
A_t = rng.integers(1, 5, (2, 3)).astype(float)  # 2×3
C_t = rng.integers(1, 5, (3, 4)).astype(float)  # 3×4
B_t = rng.integers(1, 5, (4, 2)).astype(float)  # 4×2

# Bal oldal: (ACB)^V
ACB = A_t @ C_t @ B_t                # 2×2
ACB_vec = ACB.flatten('F')            # 4×1 (oszlopok egymás alá)

# Jobb oldal: (B^T ⊗ A) @ C^V
BT_kron_A = np.kron(B_t.T, A_t)      # (2×4) ⊗ (2×3) = ... de méret: (2*2)×(3*4) = 4×12
C_vec = C_t.flatten('F')              # 12×1
result = BT_kron_A @ C_vec            # 4×1

print("Vektorizációs azonosság: (ACB)^V = (B^T ⊗ A) @ C^V")
print(f"\n(ACB)^V       = {ACB_vec}")
print(f"(B^T⊗A) @ C^V = {result}")
print(f"\nEgyeznek? {np.allclose(ACB_vec, result)}")

Vektorizációs azonosság: (ACB)^V = (B^T ⊗ A) @ C^V

(ACB)^V       = [256. 167. 347. 229.]
(B^T⊗A) @ C^V = [256. 167. 347. 229.]

Egyeznek? True


## 8. Adatok Data Wranglerben

In [16]:
df = pd.DataFrame(X, columns=['xi_1', 'xi_2', 'xi_3'])

# Centrált adatok hozzáadása
HX = H @ X
df['xi_1_cent'] = HX[:, 0]
df['xi_2_cent'] = HX[:, 1]
df['xi_3_cent'] = HX[:, 2]

# Függetlenített 3. változó
df['xi_3_given_12'] = X2_given_1[:, 0]

print(f"DataFrame: {df.shape[0]} sor × {df.shape[1]} oszlop")
print()
print(df.head(10).round(3))
print()
df

DataFrame: 200 sor × 7 oszlop

    xi_1   xi_2   xi_3  xi_1_cent  xi_2_cent  xi_3_cent  xi_3_given_12
0  4.381  3.418 -1.593      3.444      0.480      0.529         -2.333
1  1.815  2.234 -1.988      0.878     -0.703      0.134         -2.369
2  0.998  0.752 -0.991      0.061     -2.185      1.130         -1.158
3  2.201  2.559 -2.128      1.264     -0.378     -0.006         -2.577
4  2.011  2.968 -2.194      1.074      0.031     -0.072         -2.662
5 -1.906  2.838 -2.151     -2.843     -0.099     -0.029         -2.224
6  1.549  1.210 -0.141      0.612     -1.728      1.981         -0.402
7  1.309  2.597  0.414      0.372     -0.340      2.536          0.049
8  0.909  1.115 -2.665     -0.028     -1.822     -0.543         -2.855
9 -3.577  2.971 -2.946     -4.514      0.034     -0.824         -2.869



,xi_1,xi_2,xi_3,xi_1_cent,xi_2_cent,xi_3_cent,xi_3_given_12
0,4.381051,3.417624,-1.592782,3.444118,0.480320,0.528941,-2.332789
1,1.815033,2.234195,-1.988063,0.878099,-0.703109,0.133660,-2.369487
2,0.998219,0.752322,-0.991461,0.061286,-2.184982,1.130261,-1.157659
3,2.200997,2.559359,-2.127827,1.264064,-0.377945,-0.006105,-2.576647
4,2.010599,2.968480,-2.193942,1.073666,0.031176,-0.072219,-2.661605
...,...,...,...,...,...,...,...
195,1.603089,2.470023,-0.589671,0.666156,-0.467282,1.532052,-0.971986
196,4.307899,4.737641,-1.547286,3.370966,1.800337,0.574437,-2.400890
197,3.161948,3.173325,-0.860958,2.225014,0.236021,1.260765,-1.459684
198,-0.423168,1.330375,-1.615660,-1.360102,-1.606929,0.506062,-1.696056
